<a href="https://colab.research.google.com/github/UCREL/Session-6-RAG-USS-2026/blob/main/Session_6_Building_a_Basic_RAG_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building a Basic RAG System
### Indexing a corpus → Retrieval → Prompting → Evaluating responses

---

## The real-world problem we will solve

> *A clinician asks: **"Does vitamin D supplementation reduce the risk of respiratory infections?"** They need an answer that is (a) correct, (b) **grounded in published evidence**, and (c) **cites the evidence** so it can be checked.*

A plain LLM will happily answer this — sometimes correctly, sometimes with a confidently invented citation. This is unacceptable in medicine, law, finance, or any domain where being wrong is expensive.

**Retrieval-Augmented Generation (RAG)** fixes this by changing the question we ask the model. Instead of

> *"What do you know about X?"*  (tests the model's memory)

we ask

> *"Here are 5 relevant passages from the literature. Using **only** these, answer X and cite your sources."*  (tests the model's reading comprehension)

Reading comprehension is a much easier task than memorisation, so **small open models become useful**, knowledge can be **updated without retraining**, and every claim becomes **auditable**.

---

## Our data

We use **PubMedQA** (`qiaojin/PubMedQA`, subset `pqa_labeled`) — 1,000 real biomedical research questions, each paired with the abstract of the paper that answers it, an expert-written long answer, and a `yes / no / maybe` decision.

This dataset gives us **ground truth at both stages** to support comprehensive evaluations:

| Stage | Ground truth available | Metric we can compute |
|---|---|---|
| Retrieval | We know which document answers each question | Recall@k, MRR@k |
| Generation | We know the correct `yes/no/maybe` verdict | Accuracy, F1 vs. reference answer |

---

## Learning outcomes

By the end you will be able to:

1. Turn a raw corpus into a **chunked, embedded, searchable index**.
2. Implement and compare **dense**, **lexical (BM25)**, and **hybrid** retrieval.
3. Write a **grounding prompt** that forces citations and permits abstention.
4. **Measure** whether retrieval helps, using retrieval metrics *and* end-to-end answer metrics.
5. Recognise the **failure modes** of RAG and know which knob to turn for each.

---

## Agenda

| # | Step |
|---|---|
| 0 | Setup |
| 1 | The problem & the corpus |
| 2 | Chunking (indexing, part 1) |
| 3 | Embeddings & the vector index (indexing, part 2) |
| 4 | Retrieval: dense vs. BM25 vs. hybrid + **retrieval evaluation** |
| 5 | Prompting & generation |
| 6 | **Evaluating responses** (the experiment that justifies RAG) |
| 7 | Failure modes & the production stack |

Cells marked **🧪 Activity** are exercises for you to complete. Cells marked **⏭️ Optional** can be skipped if you are short on time.

> **Before you start:** `Runtime → Change runtime type → T4 GPU`. Everything runs on CPU too, just ~5× slower.

---
# Step 0 — Setup

### Why these libraries?

We deliberately start with **small, single-purpose libraries** rather than an all-in-one framework (LangChain, LlamaIndex). Frameworks hide exactly the machinery you are here to learn. Once you understand the pieces, a framework is a quick read.

| Library | Job | Why this one |
|---|---|---|
| `datasets` | Load PubMedQA | Standard HF loader; streams, caches, no manual parsing |
| `sentence-transformers` | Text → vectors | Encoder models are trained *specifically* for semantic similarity search, unlike raw BERT |
| `faiss-cpu` | Vector index | Facebook's similarity-search library; the standard baseline. Exact search on 5k vectors is instant |
| `rank_bm25` | Lexical search | 30-line implementation of the classic sparse baseline that dense retrievers still fail to beat on rare terms |
| `transformers` | Run the generator LLM | Universal interface to open-weight models |
| `accelerate` | Device placement / fp16 | Lets `transformers` use the GPU efficiently |

All open-source. All free. No API keys anywhere in this notebook.

In [ ]:
%pip install -q "sentence-transformers>=3.0" faiss-cpu rank_bm25 "datasets>=2.19" "transformers>=4.44" accelerate
print("Installed. If Colab asks you to restart the runtime, do it, then re-run from here.")

In [ ]:
import os, re, json, math, random, textwrap, warnings
from collections import defaultdict, Counter

import numpy as np
import torch

warnings.filterwarnings("ignore")

# ---------------- Reproducibility ----------------
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ---------------- Global configuration -----------
# All configuration settings are defined here.
CONFIG = {
    "embed_model":   "sentence-transformers/all-MiniLM-L6-v2",  # 22M params, 384-dim, fast on CPU
    "gen_model_gpu": "Qwen/Qwen2.5-1.5B-Instruct",              # ~3 GB in fp16 -> fits a free T4
    "gen_model_cpu": "Qwen/Qwen2.5-0.5B-Instruct",              # fallback if no GPU
    "chunk_words":   140,   # target chunk size, in words
    "chunk_overlap": 30,    # words of overlap between consecutive chunks
    "top_k":         5,     # how many chunks we feed the LLM
    "retrieve_k":    20,    # how many chunks each retriever returns before fusion/reranking
    "n_eval_retrieval": 200,  # questions used for retrieval metrics
    "n_eval_gen":       40,   # questions used for generation metrics (slower)
    "n_eval_judge":     20,   # questions used for the LLM-as-judge groundedness check
}

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU:    {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected. Everything still works, just slower.")
    print("Tip: Runtime -> Change runtime type -> T4 GPU")

---
# Step 1 — The problem and the corpus

Before writing a single line of RAG code, look at your data. **Most RAG failures are data failures**, and they are visible in the first ten rows.

### What we are building

```
                   ┌────── INDEXING (offline, once) ──────┐
  1000 abstracts → chunk → embed → FAISS index + BM25 index
                   └──────────────────────────────────────┘

                   ┌─────────── QUERY TIME (online, per question) ────────┐
  question → embed → search → top-k chunks → prompt → LLM → answer + citations
                   └──────────────────────────────────────────────────────┘
```

Each PubMedQA row contains:

* `pubid` — the PubMed ID of the source paper (**this is our ground-truth document label**)
* `question` — a real research question, e.g. *"Is anaemia associated with mortality in dialysis patients?"*
* `context.contexts` — the abstract, already split into sections (BACKGROUND / METHODS / RESULTS…)
* `long_answer` — an expert-written free-text answer (our reference for text-similarity scoring)
* `final_decision` — `yes` / `no` / `maybe` (our reference for accuracy scoring)

The `pubid` link between a question and its abstract is what lets us **evaluate retrieval quantitatively** later.

In [ ]:
from datasets import load_dataset

try:
    ds = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train")
except Exception as e:
    print("Primary load failed, trying legacy name...", e)
    ds = load_dataset("pubmed_qa", "pqa_labeled", split="train")

print(ds)
print("\n" + "=" * 90)
row = ds[0]
print("QUESTION      :", row["question"])
print("FINAL DECISION:", row["final_decision"])
print("PUBID (doc id):", row["pubid"])
print("\nABSTRACT SECTIONS:")
for lbl, ctx in zip(row["context"]["labels"], row["context"]["contexts"]):
    print(f"\n  [{lbl}]\n" + textwrap.fill(ctx, 100, initial_indent="  ", subsequent_indent="  "))
print("\nEXPERT LONG ANSWER:")
print(textwrap.fill(row["long_answer"], 100, initial_indent="  ", subsequent_indent="  "))

### Building the corpus and the query set

We now **flatten** the dataset into the two structures every RAG system needs:

* a **corpus**: a flat list of passages, each tagged with the document it came from;
* a **query set**: questions, each tagged with the `doc_id` that is known to answer it.

Note carefully: we throw away the *pairing* when we build the index. The retriever sees ~3,500 passages with no idea which question belongs to which. It has to find the right one. That is the whole game.

In [ ]:
passages, questions = [], []

for row in ds:
    doc_id = str(row["pubid"])
    for i, (lbl, ctx) in enumerate(zip(row["context"]["labels"], row["context"]["contexts"])):
        passages.append({
            "doc_id": doc_id,
            "passage_id": f"{doc_id}::{i}",
            "section": lbl,
            "text": ctx.strip(),
        })
    questions.append({
        "qid": doc_id,
        "question": row["question"].strip(),
        "gold_doc_id": doc_id,                  # <- ground truth for retrieval
        "final_decision": row["final_decision"],# <- ground truth for the answer
        "long_answer": row["long_answer"].strip(),
    })

wc = [len(p["text"].split()) for p in passages]

print(f"Passages              : {len(passages):,}")
print(f"Questions             : {len(questions):,}")
print(f"Passage length (words): min={min(wc)}  median={int(np.median(wc))}  p95={int(np.percentile(wc,95))}  max={max(wc)}")
print("Section types         :", Counter(p["section"] for p in passages).most_common(6))
print("Answer distribution   :", Counter(q["final_decision"] for q in questions))

> **Read that last line.** `yes` dominates the dataset. A model that blindly answers "yes" every time scores ~55% accuracy. **That is our baseline to beat** — and a reminder that an accuracy number means nothing until you know the majority-class rate.

---
# Step 2 — Chunking *(indexing, part 1)*

### Why chunk at all?

Three independent reasons, and it is worth to state all three:

1. **The encoder has a hard limit.** `all-MiniLM-L6-v2` truncates at **256 tokens**. Anything past that is silently discarded — you would be indexing text that no vector ever represents.
2. **Embeddings are lossy averages.** One vector for a 3,000-word document is a blurry average of everything in it. A passage about *one* idea produces a *sharp* vector. Smaller chunks → higher retrieval precision.
3. **The LLM's context is precious.** Feeding 5 whole documents wastes tokens, costs latency, and — because of the "lost in the middle" effect — actively *degrades* answer quality. Feed 5 tight passages instead.

### The trade-off

| Chunks too small | Chunks too large |
|---|---|
| Context is severed mid-argument | Vector is a blurry average → poor recall |
| The retrieved chunk lacks the sentence that qualifies it | Irrelevant text distracts the LLM |
| More vectors → slower, larger index | Encoder silently truncates |

There is no universal right answer. **The right chunk size is the one that wins on your evaluation set** — which is exactly why we build the evaluation harness in Step 4.

### Our strategy: fixed-size sliding window with overlap

PubMedQA passages are already semantically coherent (they are abstract sections), so most need no splitting. But long RESULTS sections exceed 256 tokens, so we apply a **sliding window of 140 words with 30 words of overlap**.

**Why overlap?** Without it, a sentence that straddles a boundary is destroyed — half of it lands in chunk *i*, half in chunk *i+1*, and neither vector represents the idea. Overlap is cheap insurance against splitting exactly through the answer.

**Why 140 words?** ≈ 190 tokens for biomedical English (medical terms fragment into many sub-word tokens). That leaves headroom under the encoder's 256-token limit. **Always size your chunks against your encoder's limit, not against a round number, and where possible, check against the actual tokenizer rather than an assumed ratio.**

In [ ]:
def chunk_text(text, chunk_words=140, overlap=30):
    # Sliding-window chunker over whitespace tokens (i.e., words).
    # Returns the text unchanged if it already fits in one window.
    words = text.split()
    if len(words) <= chunk_words:
        return [text]
    step = chunk_words - overlap
    assert step > 0, "overlap must be smaller than chunk_words"
    chunks = []
    for start in range(0, len(words), step):
        window = words[start:start + chunk_words]
        if len(window) < 30 and chunks:   # don't end up with a tiny fragment at the end
            break
        chunks.append(" ".join(window))
        if start + chunk_words >= len(words):
            break
    return chunks


chunks = []
for p in passages:
    for ctext in chunk_text(p["text"], CONFIG["chunk_words"], CONFIG["chunk_overlap"]):
        chunks.append({
            "chunk_id": len(chunks),
            "doc_id":   p["doc_id"],          # <- the parent link. Never lose this.
            "passage_id": p["passage_id"],
            "section":  p["section"],
            "text":     ctext,
        })

chunk_texts = [c["text"] for c in chunks]
cwc = [len(t.split()) for t in chunk_texts]
print(f"{len(passages):,} passages -> {len(chunks):,} chunks")
print(f"Chunk length (words): median={int(np.median(cwc))}  max={max(cwc)}")
print(f"Passages that were split: {sum(1 for p in passages if len(chunk_text(p['text'], CONFIG['chunk_words'], CONFIG['chunk_overlap'])) > 1):,}")
print("\nExample chunk:\n" + textwrap.fill(chunks[3]['text'], 100))

> **The single most important line above** is `"doc_id": p["doc_id"]`.
>
> We retrieve *chunks*, but we reason about *documents*. Every chunk must carry a pointer back to its parent so we can (a) evaluate against document-level ground truth, (b) show the user a real citation, and (c) fetch surrounding context if needed. A chunk with no provenance is unusable in production. This is the **metadata** half of an index, and beginners tend to forget it.

### 🧪 Activity 1 — Feel the trade-off

Run the cell below. It re-chunks the corpus at several sizes and reports how many vectors you would store and how many chunks exceed the encoder's 256-token limit.

**Predict before you run:** at `chunk_words=30`, how many chunks will there be? Is that good or bad?

In [ ]:
print(f"{'chunk_words':>12} | {'#chunks':>8} | {'median words':>12} | {'% over ~256 tok':>16}")
print("-" * 60)
for cw in [30, 60, 140, 250, 400]:
    tmp = [c for p in passages for c in chunk_text(p["text"], cw, min(CONFIG["chunk_overlap"], cw // 4))]
    lens = [len(t.split()) for t in tmp]
    over = 100 * np.mean([l * 1.35 > 256 for l in lens])   # ~1.35 tokens per biomedical word
    print(f"{cw:>12} | {len(tmp):>8,} | {int(np.median(lens)):>12} | {over:>15.1f}%")

---
# Step 3 — Embeddings & the vector index *(indexing, part 2)*

### What is an embedding?

A function that maps text to a point in $\mathbb{R}^{384}$ such that **texts that mean similar things land near each other**, even when they share no words.

That last clause is the whole reason dense retrieval exists. The query *"Does taking vitamin D stop you getting colds?"* and the passage *"Cholecalciferol supplementation reduced acute respiratory tract infection incidence"* have **almost zero word overlap**. Keyword search scores them at ~0. A well-trained encoder places them close together.

### Why `sentence-transformers/all-MiniLM-L6-v2`?

* **It is trained for this task.** Raw BERT embeddings are *bad* at similarity search — BERT was trained to predict masked words, not to make similar sentences neighbours. MiniLM was fine-tuned contrastively on ~1B sentence pairs specifically so that cosine similarity means semantic similarity. Using raw BERT here is a classic beginner error.
* **22M parameters, 384 dimensions.** Encodes our ~3,500 chunks quickly on typical hardware.
* **Ungated, Apache-2.0.** No token, no licence click-through.

Bigger encoders (`BAAI/bge-base-en-v1.5`, `intfloat/e5-base-v2`, `nomic-embed-text-v1.5`) score higher on the [MTEB leaderboard](https://huggingface.co/spaces/mteb/leaderboard) and are drop-in replacements — Activity 6 invites you to try one.

### Cosine similarity, cheaply

$$\text{Sim}(A, B) = \frac{A.B}{||A||||B||}$$

We `normalize_embeddings=True`, which makes every vector unit-length. For unit vectors, the **dot product equals the cosine similarity**. So we can use FAISS's `IndexFlatIP` (Inner Product) and get cosine ranking for free.

### Why FAISS `IndexFlatIP` and not an approximate index?

`Flat` = brute force: compare the query against every vector. It gives **exact** results - no approximation, no accuracy trade-off. And, at our scale (3,480 vectors × 384 dimensions), a brute force search over every vector is still instantaneous.

Approximate indexes like `IVF`(clusters vectors into buckets, only searches nearby buckets) or `HNSW` (builds a navigable graph structure to skip most comparisons) exist for a different situation: once you have millions of vectors, brute-force search becomes too slow, so these indexes sacrifice a small amount of accuracy in exchange for much faster search. However, **don't reach for approximate search before you need it** — you would be introducing an unmeasured accuracy loss to solve a problem you don't have.

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(CONFIG["embed_model"], device=DEVICE)
print("Embedding dim   :", embedder.get_sentence_embedding_dimension())
print("Max seq length  :", embedder.max_seq_length, "tokens  <- this is why we chunked")

chunk_vecs = embedder.encode(
    chunk_texts,
    batch_size=64,
    convert_to_numpy=True,
    normalize_embeddings=True,     # unit vectors -> dot product == cosine
    show_progress_bar=True,
).astype("float32")

print("\nMatrix shape:", chunk_vecs.shape, f"({chunk_vecs.nbytes/1e6:.1f} MB)")

In [ ]:
import faiss

index = faiss.IndexFlatIP(chunk_vecs.shape[1])   # IP == cosine, because vectors are normalised
index.add(chunk_vecs)
print("Vectors indexed:", index.ntotal)


def dense_search(query, k=10):
    # Returns [(chunk_id, score), ...] ranked by cosine similarity.
    qv = embedder.encode([query], convert_to_numpy=True,
                         normalize_embeddings=True).astype("float32")
    scores, ids = index.search(qv, k)
    return list(zip(ids[0].tolist(), scores[0].tolist()))


def show(results, n=3, width=95):
    for rank, (cid, score) in enumerate(results[:n], 1):
        c = chunks[cid]
        print(f"\n#{rank}  score={score:.3f}  doc={c['doc_id']}  section={c['section']}")
        print(textwrap.fill(c["text"][:400] + "...", width, initial_indent="    ", subsequent_indent="    "))

In [ ]:
q = "Does vitamin D supplementation reduce the risk of respiratory infections?"
print("QUERY:", q)
show(dense_search(q, k=3))

### 🧪 Activity 2 — Prove that it is *semantic*, not keyword matching

Run the cell below with Query A — written in **plain English with no biomedical vocabulary**. It prints the word overlap between the query and the top hit.

If overlap is near zero but the passage is on-topic, the encoder is doing real semantic work, not keyword matching.

Now switch to Query B — a colloquial paraphrase using invented slang ("sunshine pill") instead of a real drug name. What do you observe?

**Then try to break it.** Edit `q2` and hunt for more failure cases. Good hunting grounds:
* A **rare identifier**: a specific gene name, a drug code, an acronym the encoder never saw.
* **Negation**: *"trials where the drug did **not** work"* — embeddings are notoriously bad at negation.
* A **numeric constraint**: *"studies with more than 500 participants"*.

Whatever you find, remember it — it is exactly the gap BM25 fills in Step 4.

In [ ]:
def overlap_ratio(a, b):
    A = set(re.findall(r"[a-z]{4,}", a.lower()))
    B = set(re.findall(r"[a-z]{4,}", b.lower()))
    return len(A & B) / max(len(A), 1)

query_a = "Does taking vitamin D stop you getting colds?"
query_b = "can taking a sunshine pill stop you catching a cold in winter"

q2 = query_a   # <-- try query_a, then query_b, then write your own

res = dense_search(q2, k=3)
print("QUERY:", q2)
print(f"Word overlap with top hit: {overlap_ratio(q2, chunks[res[0][0]]['text']):.1%}")
show(res, n=2)

---
# Step 4 — Retrieval: dense vs. lexical vs. hybrid
*(The most important step in the notebook)*

Everything downstream is capped by retrieval. **If the answer is not in the retrieved chunks, no prompt and no model can save you.** So this is where you spend your engineering effort.

### The two families

1. **Dense (bi-encoder + vector search)** — matches *meaning*. Robust to paraphrase and synonymy. Blind to terms outside its training distribution: a brand-new drug name is, to the encoder, semantic noise.

2. **Lexical (BM25)** — matches *words*, weighting rare words heavily (IDF) and saturating on repetition. Exact by construction, so it nails identifiers, acronyms and jargon (specialised terminology). Utterly defeated by paraphrase.

Their errors are **decorrelated**, which is precisely the condition under which combining two systems beats both. Hybrid retrieval is not a hack; it is the standard configuration in serious production systems.

### Fusing them: Reciprocal Rank Fusion (RRF)

We cannot add a cosine similarity (0–1) to a BM25 score (unbounded, corpus-dependent) — the scales are meaningless relative to each other. Score normalisation is fragile and needs tuning.

RRF sidesteps this entirely by **throwing away the scores and keeping only the ranks**:

$$\text{RRF}(d) = \sum_{r \in \text{retrievers}} \frac{1}{k + \text{rank}_r(d)}, \qquad k = 60$$

A document ranked #1 by either retriever gets a big boost; a document ranked highly by *both* wins. It has **no parameters to tune** (k=60 is the standard value from the original paper and is remarkably insensitive), and it works across any mixture of retrievers. Start here; only move to a learned fusion if evaluation says you need to.

In [ ]:
from rank_bm25 import BM25Okapi

STOP = set("a an and are as at be by for from has have in is it its of on or that the to was were with we this these those".split())

def tokenize(text):
    return [w for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in STOP]

bm25 = BM25Okapi([tokenize(t) for t in chunk_texts])
print("BM25 index built over", len(chunk_texts), "chunks")


def bm25_search(query, k=10):
    scores = bm25.get_scores(tokenize(query))
    top = np.argsort(-scores)[:k]  # sort in descending order
    return [(int(i), float(scores[i])) for i in top]


def rrf_fuse(*rankings, k=60, top_k=10):
    # Reciprocal Rank Fusion. Input: several [(chunk_id, score), ...] lists.
    # Scores are discarded; only the ranks matter.
    fused = defaultdict(float)
    for ranking in rankings:
        for rank, (cid, _) in enumerate(ranking, start=1):
            fused[cid] += 1.0 / (k + rank)
    return sorted(fused.items(), key=lambda x: -x[1])[:top_k]


def hybrid_search(query, k=10, pool=None):
    pool = pool or CONFIG["retrieve_k"]
    return rrf_fuse(dense_search(query, pool), bm25_search(query, pool), top_k=k)

In [ ]:
# Where the two retrievers disagree
probe = "CD14 polymorphism and asthma risk in children"     # rare identifier: BM25 should shine
print("QUERY:", probe, "\n")

for name, fn in [("DENSE ", dense_search), ("BM25  ", bm25_search), ("HYBRID", hybrid_search)]:
    docs = [chunks[cid]["doc_id"] for cid, _ in fn(probe, k=5)]
    print(f"{name} top-5 doc_ids: {docs}")

print("\nNotice how the three lists differ. Now the question that matters:")
print("which ordering actually puts the CORRECT document at the top, on average?")

### Evaluating retrieval — the step that is often ignored

We have three retrievers and three opinions. Opinions do not scale. **Measure** instead.

Our ground truth: for question `q`, the relevant chunks are exactly those whose `doc_id == q["gold_doc_id"]`.

Two metrics, and it is worth knowing why we need both:

**Recall@k** — *"Is the right document anywhere in the top k?"*
This is the **ceiling on your whole system**. If Recall@5 = 0.70, then 30% of your questions are unanswerable no matter how good your LLM is. It is a hard upper bound and the first number you should ever compute.

**MRR@k** (Mean Reciprocal Rank) — *"How high up is the right document?"*
$\text{MRR} = \frac{1}{|Q|}\sum_q \frac{1}{\text{rank of first relevant chunk}}$

Recall is blind to ordering; MRR is not. Position matters because (a) you usually truncate at small *k*, and (b) LLMs attend more strongly to the beginning and end of their context — the *"lost in the middle"* effect. A retriever that returns the answer at rank 5 is measurably worse than one that returns it at rank 1, even at identical recall.

We evaluate on a random sample of **200 questions**, given the time constraints.

In [ ]:
def evaluate_retriever(search_fn, qs, k=5, name=""):
    recall, rr = [], []
    for q in qs:
        results = search_fn(q["question"], k=k)
        hit_ranks = [i for i, (cid, _) in enumerate(results, 1)
                     if chunks[cid]["doc_id"] == q["gold_doc_id"]]
        recall.append(1.0 if hit_ranks else 0.0)
        rr.append(1.0 / hit_ranks[0] if hit_ranks else 0.0)
    return {"retriever": name, f"Recall@{k}": np.mean(recall), f"MRR@{k}": np.mean(rr)}


random.seed(SEED)
eval_qs = random.sample(questions, CONFIG["n_eval_retrieval"])
K = CONFIG["top_k"]

rows = [
    evaluate_retriever(dense_search,  eval_qs, K, "Dense (MiniLM)"),
    evaluate_retriever(bm25_search,   eval_qs, K, "BM25 (lexical)"),
    evaluate_retriever(hybrid_search, eval_qs, K, "Hybrid (RRF)"),
]

print(f"Evaluated on {len(eval_qs)} questions, k={K}\n")
print(f"{'retriever':<18} {'Recall@'+str(K):>10} {'MRR@'+str(K):>10}")
print("-" * 40)
for r in rows:
    print(f"{r['retriever']:<18} {r[f'Recall@{K}']:>10.3f} {r[f'MRR@{K}']:>10.3f}")

BEST = max(rows, key=lambda r: r[f"MRR@{K}"])["retriever"]
print(f"\nWinner by MRR: {BEST}")
print("\nInterpretation: Recall@k is the ceiling on end-to-end accuracy.")
print("Whatever fraction of questions are missing here is permanently lost downstream.")

### 🧪 Activity 3 — Experimenting `k`

Evaluate the hybrid retriever at `k ∈ {1, 3, 5, 10, 20}`. Recall rises monotonically with `k` — so why not just set `k=20`? (Hint: what are you about to paste into the LLM's context window? What does that do to cost, latency, and the lost-in-the-middle effect?)

Use the code below.

In [ ]:
# 🧪 Your ablation here. Example: the k-sweep.
print(f"{'k':>4} {'Recall@k':>10} {'MRR@k':>8}")
print("-" * 24)
for k in [1, 3, 5, 10, 20]:
    r = evaluate_retriever(hybrid_search, eval_qs[:100], k=k, name="hybrid")
    print(f"{k:>4} {r[f'Recall@{k}']:>10.3f} {r[f'MRR@{k}']:>8.3f}")

# Now ask yourself: recall keeps climbing. Why is k=20 still a bad idea in production?

### ⏭️ Optional — Cross-encoder reranking (the single best accuracy-per-line-of-code upgrade)

Our retrievers are **bi-encoders**: query and chunk are embedded *independently*, then compared. That independence is what makes the index precomputable and search fast — but it means the model never sees the query and the chunk *together*.

A **cross-encoder** does: it feeds `[query] [SEP] [chunk]` through a transformer jointly and outputs one relevance score. Full cross-attention between question and passage makes it dramatically more accurate — and far too slow to run over 3,480 chunks.

So we use the standard two-stage architecture:

```
retrieve 20 candidates (fast, approximate)  →  rerank to 5 (slow, precise)
```

20 cross-encoder passes per query is cheap. Recall is set by stage 1; **precision at the top is set by stage 2**.

In [ ]:
# Run the following code to observe the impact of a cross-encoder

from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=DEVICE, max_length=384)

def rerank_search(query, k=5, pool=None):
    pool = pool or CONFIG["retrieve_k"]
    cands = [cid for cid, _ in hybrid_search(query, k=pool)]
    scores = reranker.predict([(query, chunks[c]["text"]) for c in cands], show_progress_bar=False)
    order = np.argsort(-scores)[:k]
    return [(cands[i], float(scores[i])) for i in order]

sub = eval_qs[:100]
print(f"{'retriever':<22} {'Recall@5':>10} {'MRR@5':>8}")
print("-" * 42)
for nm, fn in [("Hybrid (RRF)", hybrid_search), ("Hybrid + reranker", rerank_search)]:
    r = evaluate_retriever(fn, sub, 5, nm)
    print(f"{nm:<22} {r['Recall@5']:>10.3f} {r['MRR@5']:>8.3f}")

In [ ]:
# Lock in the retriever the rest of the notebook will use.
RETRIEVER = hybrid_search        # swap to rerank_search if you ran the optional cell
print("Using:", RETRIEVER.__name__)

---
# Step 5 — Prompting & generation

Retrieval found the evidence. The LLM's job is now narrow and mechanical: **read these 5 passages and answer using only them.**

### Choosing the generator

`Qwen/Qwen2.5-1.5B-Instruct` — 1.5B parameters, ~3 GB in fp16, comfortable on a free T4, Apache-2.0, ungated, and unusually good at instruction-following for its size.

Note what we are *not* doing: we are not choosing a model for its biomedical knowledge. **In RAG, the knowledge lives in the corpus.** We are selecting for reading comprehension, instruction-following, and format compliance. This is why a 1.5B model with good retrieval routinely beats a 70B model without it — and it is the central economic argument for RAG.

### Anatomy of a grounding prompt

Four elements, each doing a specific job:

| Element | Purpose |
|---|---|
| **Role + scope** (*"answer only from the excerpts"*) | Suppresses parametric memory; makes the corpus the sole authority |
| **Numbered excerpts** `[1] [2] [3]` | Gives the model a stable handle to cite |
| **An explicit escape hatch** (*"if the excerpts don't say, answer `maybe`"*) | **Abstention.** Without permission to say "I don't know", a model *will* invent. This single instruction is the biggest hallucination reduction available to you |
| **A rigid output schema** | `DECISION / ANSWER / CITATIONS` — parseable output turns a chatbot into a component you can automate and, crucially, **evaluate** |

We also set `do_sample=False` (greedy decoding). Sampling is for creative writing. For factual QA — and above all for **reproducible evaluation** — you want the same input to yield the same output.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

GEN_NAME = CONFIG["gen_model_gpu"] if DEVICE == "cuda" else CONFIG["gen_model_cpu"]
print("Loading generator:", GEN_NAME)

tok = AutoTokenizer.from_pretrained(GEN_NAME)
gen_model = AutoModelForCausalLM.from_pretrained(
    GEN_NAME,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map="auto" if DEVICE == "cuda" else None,
)
if DEVICE != "cuda":
    gen_model = gen_model.to("cpu")
gen_model.eval()

if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "left"          # required for correct batched generation with decoder-only models

print(f"Loaded {sum(p.numel() for p in gen_model.parameters())/1e9:.2f}B parameters")

In [ ]:
SYSTEM = (
    "You are a careful biomedical research assistant. You answer strictly from the "
    "excerpts you are given. You never use outside knowledge. If the excerpts do not "
    "settle the question, you say so rather than guessing."
)

SYSTEM_CLOSED_BOOK = (
    "You are a careful biomedical research assistant answering from memory alone."
)

INSTRUCTIONS = (
    "Answer the question using ONLY the excerpts above.\n"
    "Reply in EXACTLY this format, with no extra text:\n"
    "DECISION: yes | no | maybe\n"
    "ANSWER: <two or three sentences>\n"
    "CITATIONS: [n], [m]\n\n"
    "Rules:\n"
    "- Use 'maybe' if the excerpts are inconclusive or do not address the question.\n"
    "- Every claim in ANSWER must be supported by a cited excerpt.\n"
    "- Cite only excerpt numbers that you actually used."
)

INSTRUCTIONS_CLOSED_BOOK = (
    "Answer the question from your own knowledge. You have no excerpts.\n"
    "Reply in EXACTLY this format, with no extra text:\n"
    "DECISION: yes | no | maybe\n"
    "ANSWER: <two or three sentences>\n"
    "CITATIONS: none\n\n"
    "Rules:\n"
    "- Use 'maybe' if you are not confident."
)

def build_prompt(question, ctx_chunks, closed_book=False):
    if closed_book:
        user = f"Question: {question}\n\n{INSTRUCTIONS_CLOSED_BOOK}"
    else:
        blocks = "\n\n".join(
            f"[{i}] (doc {chunks[cid]['doc_id']}, {chunks[cid]['section']})\n{chunks[cid]['text']}"
            for i, cid in enumerate(ctx_chunks, 1)
        )
        user = f"EXCERPTS:\n{blocks}\n\nQuestion: {question}\n\n{INSTRUCTIONS}"
    sys_msg = SYSTEM_CLOSED_BOOK if closed_book else SYSTEM
    return [{"role": "system", "content": sys_msg}, {"role": "user", "content": user}]


@torch.no_grad()
def generate(prompts, max_new_tokens=140):
    # Batched greedy generation. `prompts` is a list of chat-message lists.
    texts = [tok.apply_chat_template(p, tokenize=False, add_generation_prompt=True) for p in prompts]
    enc = tok(texts, return_tensors="pt", padding=True, truncation=True, max_length=3072).to(gen_model.device)
    out = gen_model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tok.pad_token_id)
    return tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)


def parse(raw, n_ctx=0):
    # Turn the model's text into a structured record. Robust to minor format drift.
    d = re.search(r"DECISION:\s*(yes|no|maybe)", raw, re.I)
    a = re.search(r"ANSWER:\s*(.+?)(?=\nCITATIONS:|\Z)", raw, re.S | re.I)
    c = re.search(r"CITATIONS:\s*(.*)", raw, re.I)
    cites = [int(x) for x in re.findall(r"\[(\d+)\]", c.group(1))] if c else []
    return {
        "decision": d.group(1).lower() if d else "maybe",      # default to abstention if unparseable
        "answer":   a.group(1).strip() if a else raw.strip(),
        "citations": cites,
        "citations_valid": all(1 <= i <= n_ctx for i in cites) and len(cites) > 0 if n_ctx else None,
        "raw": raw,
    }

In [ ]:
def rag_answer(question, k=None, verbose=True):
    k = k or CONFIG["top_k"]
    hits = RETRIEVER(question, k=k)
    ctx = [cid for cid, _ in hits]
    raw = generate([build_prompt(question, ctx)])[0]
    out = parse(raw, n_ctx=len(ctx))
    out["context_chunks"] = ctx
    out["context_docs"] = [chunks[c]["doc_id"] for c in ctx]

    if verbose:
        print("QUESTION :", question)
        print("RETRIEVED:", out["context_docs"])
        print("-" * 90)
        print("DECISION :", out["decision"].upper())
        print("ANSWER   :", textwrap.fill(out["answer"], 90, subsequent_indent="           "))
        print("CITATIONS:", out["citations"], "-> valid:", out["citations_valid"])
        if out["citations"]:
            print("\nEvidence actually cited:")
            for i in out["citations"]:
                if 1 <= i <= len(ctx):
                    c = chunks[ctx[i - 1]]
                    print(f"  [{i}] doc {c['doc_id']} ({c['section']}): {c['text'][:200]}...")
    return out


demo = questions[0]
_ = rag_answer(demo["question"])
print("\n" + "=" * 90)
print("GROUND TRUTH DECISION:", demo["final_decision"].upper())
print("EXPERT ANSWER:", textwrap.fill(demo["long_answer"], 90, subsequent_indent="  "))

### 🧪 Activity 4 — Try to make it hallucinate

The abstention clause is the safety rail. **Test it.**

1. Ask something the corpus cannot possibly answer: *"Is Lima the capital of Peru?"*, *"Does drinking coffee cure baldness?"*. Retrieval will return its 5 nearest, but irrelevant, chunks. **Does the model say `maybe`, or does it confabulate from unrelated text?**
2. **Read the `ANSWER` text closely.** Do you see any contradiction?

**This is a core lesson of prompting for RAG:** citations could make an answer look grounded, although it is not.


In [ ]:
# 🧪 Adversarial probes. Edit freely.
for probe in ["Is Lima the capital of Peru?",
              "Does drinking coffee cure baldness?"]:
    print("\n" + "=" * 90)
    rag_answer(probe)

---
# Step 6 — Evaluating responses
*(The step that separates a demo from a system)*

A RAG demo always looks impressive. You type a question, you get a fluent paragraph with a citation. However, this does not prove the system is working.

We need to focus on two critical questions:

1. Does retrieval **actually help**, or is the model answering from memory, with citations giving the illusion of retrieval?
2. When the model is confidently wrong, **would we notice**?

Both are answerable, and both need controlled experiments.

### The controlled experiment

We run the same 40 questions through three conditions:

| Condition | Context given | Tests |
|---|---|---|
| **Closed-book** | none | What the model already knows |
| **RAG** | top-5 retrieved chunks | What retrieval adds |
| **Distractor** | top-5 chunks for a *different, random* question | Whether the model is genuinely reading, or just pattern-matching the question |

The **distractor** is the easiest to overlook, yet it is the most diagnostic. If distractor-accuracy ≈ RAG-accuracy, your model is ignoring the context and answering from parametric memory — your retrieval pipeline is expensive decoration. If distractor-accuracy collapses toward chance *and abstention rises*, the model is genuinely grounded.

### What we measure

* **Decision accuracy** vs. `final_decision` — against the ~55% majority-class baseline.
* **Abstention rate** (`maybe`) — under distractors, *rising* abstention is a **good** sign. A model that says "I don't know" when handed irrelevant evidence is behaving correctly. Never optimise abstention to zero.
* **Token-F1** vs. the expert `long_answer` — a simple but honest, reference-based overlap score. It cannot see meaning, only words; treat it as a smoke detector, not a thermometer.
* **Citation validity** — are the cited excerpt numbers real, and does at least one point at the gold document? A model can cite `[7]` when you gave it 5 excerpts. Check.

In [ ]:
def token_f1(pred, ref):
    # Unigram overlap F1 -- the standard SQuAD-style reference metric.
    p = Counter(tokenize(pred)); r = Counter(tokenize(ref))
    common = sum((p & r).values())
    if common == 0:
        return 0.0
    prec, rec = common / max(sum(p.values()), 1), common / max(sum(r.values()), 1)
    return 2 * prec * rec / (prec + rec)


random.seed(SEED + 1)
gen_qs = random.sample(questions, CONFIG["n_eval_gen"])
K = CONFIG["top_k"]

# Pre-compute the three context conditions for every question.
conditions = {"closed_book": [], "rag": [], "distractor": []}
for i, q in enumerate(gen_qs):
    good = [cid for cid, _ in RETRIEVER(q["question"], k=K)]
    other = gen_qs[(i + 7) % len(gen_qs)]                 # a deterministic "different" question
    bad = [cid for cid, _ in RETRIEVER(other["question"], k=K)]
    bad = [c for c in bad if chunks[c]["doc_id"] != q["gold_doc_id"]][:K]   # guarantee no gold leak
    conditions["closed_book"].append([])
    conditions["rag"].append(good)
    conditions["distractor"].append(bad)

print(f"Prepared {len(gen_qs)} questions x 3 conditions. Generating (this is the slow part)...")

In [ ]:
BATCH = 8 if DEVICE == "cuda" else 2
results = {}

for cond, ctxs in conditions.items():
    prompts = [build_prompt(q["question"], c, closed_book=(cond == "closed_book"))
               for q, c in zip(gen_qs, ctxs)]
    raws = []
    for i in range(0, len(prompts), BATCH):
        raws += generate(prompts[i:i + BATCH])
        print(f"  {cond:<12} {min(i+BATCH, len(prompts))}/{len(prompts)}", end="\r")
    results[cond] = [parse(r, n_ctx=len(c)) for r, c in zip(raws, ctxs)]
    print(f"  {cond:<12} done{' '*20}")

print("\nGeneration complete.")

In [ ]:
majority = Counter(q["final_decision"] for q in gen_qs).most_common(1)[0]
print(f"Majority-class baseline: always answer '{majority[0]}' -> {majority[1]/len(gen_qs):.1%} accuracy\n")

print(f"{'condition':<14} {'accuracy':>9} {'abstain(maybe)':>15} {'token-F1':>10} {'cites valid':>12} {'cites gold':>11}")
print("-" * 78)

summary = {}
for cond, outs in results.items():
    acc = np.mean([o["decision"] == q["final_decision"] for o, q in zip(outs, gen_qs)])
    absn = np.mean([o["decision"] == "maybe" for o in outs])
    f1 = np.mean([token_f1(o["answer"], q["long_answer"]) for o, q in zip(outs, gen_qs)])

    if cond == "closed_book":
        cv = cg = float("nan")
    else:
        cv = np.mean([bool(o["citations_valid"]) for o in outs])
        cg = np.mean([
            any(chunks[ctx[i - 1]]["doc_id"] == q["gold_doc_id"]
                for i in o["citations"] if 1 <= i <= len(ctx))
            for o, q, ctx in zip(outs, gen_qs, conditions[cond])
        ])
    summary[cond] = dict(acc=acc, abstain=absn, f1=f1, cite_valid=cv, cite_gold=cg)
    print(f"{cond:<14} {acc:>8.1%} {absn:>14.1%} {f1:>10.3f} {cv:>11.1%} {cg:>10.1%}")

print("\n" + "=" * 78)
print("HOW TO READ THIS TABLE")
print("=" * 78)
print("1. RAG accuracy > closed-book accuracy  -> retrieval adds real information.")
print("2. Distractor accuracy << RAG accuracy  -> the model is reading the context.")
print("   If they are close, your model is answering from memory and RAG is theatre.")
print("3. Abstention RISES under distractors   -> the abstention clause is working.")
print("4. 'cites gold' is the honest number: how often does the model point at the")
print("   document that actually answers the question? This is auditability.")
print("\nSmall models could be noisy at n=40. Raise CONFIG['n_eval_gen'] if you have time.")

### ⏭️ Optional — LLM-as-judge: measuring *groundedness*

Accuracy tells us whether the verdict was right. It does **not** tell us whether the *reasoning* was supported by the evidence — a model can reach the right `yes` for entirely invented reasons. That property is called **groundedness** (or *faithfulness*): is every claim in the answer entailed by the retrieved context?

There is no reference answer for groundedness, so we use a second LLM as a judge, giving it the context and the answer and asking for a verdict.

> ⚠️ **Be honest about the limits of this method.** We are judging a 1.5B model with the *same* 1.5B model. That is a weak judge, and it is **self-preferring** — models systematically rate their own outputs favourably. In real work you use a stronger, *different* judge, you validate it against a few hundred human labels, and you report the judge–human agreement rate alongside the score. Without validation, an LLM judge is just another opinion—only expressed as a score.

> We run it here because the *mechanism* is what you need to understand, and because it is what libraries like **RAGAS**, **TruLens** and **DeepEval** automate for you.

In [ ]:
JUDGE = (
    "You are a strict fact-checking judge. You will see EXCERPTS and an ANSWER.\n"
    "Decide whether EVERY factual claim in the ANSWER is supported by the EXCERPTS.\n"
    "Outside knowledge does not count as support. Reply with exactly one word: "
    "SUPPORTED or UNSUPPORTED."
)

def judge_batch(outs, ctxs, n):
    prompts = []
    for o, ctx in list(zip(outs, ctxs))[:n]:
        blocks = "\n\n".join(f"[{i}] {chunks[c]['text']}" for i, c in enumerate(ctx, 1))
        prompts.append([
            {"role": "system", "content": JUDGE},
            {"role": "user", "content": f"EXCERPTS:\n{blocks}\n\nANSWER:\n{o['answer']}\n\nVerdict:"},
        ])
    verdicts = []
    for i in range(0, len(prompts), BATCH):
        verdicts += generate(prompts[i:i + BATCH], max_new_tokens=6)
    return ["SUPPORTED" if v.strip().upper().startswith("SUPPORTED") else "UNSUPPORTED" for v in verdicts]

N = CONFIG["n_eval_judge"]
print(f"Judging {N} answers per condition...\n")
for cond in ["rag", "distractor"]:
    v = judge_batch(results[cond], conditions[cond], N)
    print(f"{cond:<12} groundedness = {v.count('SUPPORTED')/len(v):>6.1%}")

print("\nExpected: RAG answers score high, distractor answers score low.")
print("If distractor answers score HIGH, the judge is failing to distinguish good answers\n from bad ones—which is precisely why an unvalidated judge is worse than no judge at all.")

### 🧪 Activity 5 — Inspect the failures

Aggregated numbers hide everything interesting. **Read the errors.**

Run the cell below to print questions the RAG system got wrong. For each, classify the failure into exactly one bucket:

| Bucket | Symptom | The fix |
|---|---|---|
| **Retrieval miss** | Gold `doc_id` is not in `context_docs` | Better encoder, hybrid, larger `k`, different chunking |
| **Reading failure** | Gold doc *was* retrieved; model still wrong | Bigger generator, better prompt, few-shot examples |
| **Genuine ambiguity** | Even a human would say "maybe" | Nothing — the label is arguable |

Tally the buckets across the group. **The bucket that dominates tells you where the next hour of engineering should go.** This diagnosis-first habit is one of the most valuable things to take away from today, because it stops you from tuning prompts when your problem is retrieval (or vice versa).

In [ ]:
errors = [(q, o, ctx) for q, o, ctx in zip(gen_qs, results["rag"], conditions["rag"])
          if o["decision"] != q["final_decision"]]
print(f"{len(errors)}/{len(gen_qs)} RAG answers were wrong. Showing the first 5:\n")

for q, o, ctx in errors[:5]:
    retrieved_gold = q["gold_doc_id"] in [chunks[c]["doc_id"] for c in ctx]
    bucket = "RETRIEVAL MISS" if not retrieved_gold else "READING FAILURE (or ambiguity)"
    print("=" * 95)
    print("Q         :", textwrap.fill(q["question"], 95, subsequent_indent="            "))
    print("Gold      :", q["final_decision"].upper(), " | Predicted:", o["decision"].upper())
    print("Gold doc retrieved?", retrieved_gold, " ->", bucket)
    print("Model said:", textwrap.fill(o["answer"], 95, subsequent_indent="            "))
    print("Expert    :", textwrap.fill(q["long_answer"][:300], 95, subsequent_indent="            "))

---
# Step 7 — Failure modes & the production stack

### A diagnostic table to take home

When your RAG system misbehaves, **do not start by editing the prompt.** Localise the fault first — that is what the metrics you just built are for.

| Symptom | Check this metric | Likely cause | Fix |
|---|---|---|---|
| Answers are confidently wrong | Recall@k | Right doc never retrieved | Hybrid retrieval, better encoder, larger `k`, re-chunk |
| Right doc retrieved, wrong answer | MRR@k, groundedness | Gold buried at rank 5; lost-in-the-middle | Add a cross-encoder reranker |
| Model ignores the context | Distractor accuracy ≈ RAG accuracy | Parametric memory dominating | Strengthen the grounding instruction; use a model tuned for it |
| Invents citations | Citation validity | No format constraint | Rigid output schema + programmatic validation; reject and retry |
| Never says "I don't know" | Abstention rate ≈ 0 | No escape hatch in the prompt | Add explicit abstention permission; consider a relevance threshold |
| Great in demo, bad in production | *(no metric)* | You never built an eval set | Label 100 questions, ideally pulled from actual user queries/logs. This is not optional |


### What we deliberately left out

* **Query transformation** — HyDE, multi-query expansion, query rewriting for follow-up turns in a conversation.
* **Metadata filtering** — restrict to `year > 2020` or `section == "RESULTS"` *before* vector search. Cheap, and often the largest single win.
* **Semantic / structure-aware chunking** — split on document structure or topic shifts rather than word counts.
* **Approximate indexes** — HNSW / IVF-PQ, once you pass ~1M vectors.
* **Multi-hop RAG** — questions needing evidence combined across several documents.

### The production stack

| Layer | Options |
|---|---|
| Orchestration | LangChain, LlamaIndex, Haystack |
| Vector DB | Qdrant, Weaviate, Milvus, Chroma, pgvector |
| Embeddings | BGE, E5, GTE, Nomic, Jina *(all open, all on the MTEB leaderboard)* |
| Reranker | bge-reranker-v2-m3, Cohere Rerank, ColBERT |
| Serving | vLLM, Ollama, llama.cpp, TGI |
| Evaluation | RAGAS, TruLens, DeepEval, Phoenix |

Everything you built today is a component in that stack. You now know what each layer is actually doing — and, more usefully, **how to tell whether it is doing it well.**

In [ ]:
# NOTE: this cell waits for keyboard input. If you used "Run all", it will pause here.
# Press Enter on an empty line to exit.
# Free-play: ask your own questions.
# Try one you think the corpus covers, and one you know it doesn't.
while True:
    q = input("\nYour question (blank to quit): ").strip()
    if not q:
        print("Done.")
        break
    print()
    rag_answer(q)

### 🧪 Activity 6 — Expanding experiments *(optional — take-home)*

> Each of these requires re-running Steps 2–3 (re-embedding the whole corpus, and in one case downloading a larger encoder).

Try different settings and see how they can help:

1. **Chunk size.** Set `CONFIG["chunk_words"] = 60`, re-run Steps 2–3, and re-evaluate. Did recall go up or down? Why?
2. **Better encoder.** Set `CONFIG["embed_model"] = "BAAI/bge-base-en-v1.5"`, re-run Step 3, re-evaluate. How much recall does 5× the parameters buy you?
4. **Fusion depth.** In `hybrid_search`, change `pool` from 20 to 50. Does giving RRF a deeper pool help?

---
# Recap

You built, end to end:

1. **Indexing** — chunked ~1,000 abstracts with an overlapping sliding window sized against the encoder's token limit, embedded them with a contrastively-trained bi-encoder, and stored them in an exact FAISS index. *Every chunk kept a pointer to its parent document.*
2. **Retrieval** — dense, lexical (BM25), and a hybrid fused by Reciprocal Rank Fusion; optionally reranked by a cross-encoder. *Then you measured all of them.*
3. **Prompting** — a grounding prompt with numbered excerpts, a rigid output schema, mandatory citations, and an explicit abstention clause.
4. **Evaluation** — Recall@k and MRR@k for retrieval; a three-arm controlled experiment (closed-book / RAG / distractor) plus citation validation and LLM-as-judge groundedness for generation.

### The three ideas worth remembering

> **1. Retrieval is the ceiling.** Recall@k bounds end-to-end accuracy. No prompt recovers a document that was never retrieved.
>
> **2. Abstention is a feature.** A model permitted to say "I don't know" is a model that hallucinates less. Never tune abstention to zero.
>
> **3. Without the distractor arm, you cannot claim RAG works.** Anyone can show a demo. Only the controlled comparison shows that the model is reading rather than remembering.

### Try next
* Swap the corpus for your own PDFs (`pypdf` → `chunk_text` → same pipeline).
* Add metadata filters, then re-measure Recall@k.
* Replace the judge with a stronger model and check its agreement with your own labels on 50 examples.

### References
* Lewis et al. (2020), *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks* — the original RAG paper.
* Jin et al. (2019), *PubMedQA: A Dataset for Biomedical Research Question Answering*.
* Cormack et al. (2009), *Reciprocal Rank Fusion Outperforms Condorcet and Individual Rank Learning Methods*.
* Liu et al. (2023), *Lost in the Middle: How Language Models Use Long Contexts*.
* Robertson & Zaragoza (2009), *The Probabilistic Relevance Framework: BM25 and Beyond*.
* MTEB leaderboard — https://huggingface.co/spaces/mteb/leaderboard